In [ ]:
!pip install -q torch torchvision tqdm pandas scikit-learn
import warnings, torch
warnings.filterwarnings('ignore', category=UserWarning)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Config
> Adjust the paths to match your dataset slugs on Kaggle.

In [ ]:
import os, numpy as np
import torch

TRAIN_CSV      = '/kaggle/input/datasets/nejikhouja/this-is-the-split-data/train.csv'
IMAGE_ROOT     = '/kaggle/input/datasets/organizations/nih-chest-xrays/data'
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE   = 32      
NUM_EPOCHS   = 40
NUM_WORKERS  = 4
VAL_SPLIT    = 0.1     

# backbone (pretrained) needs much smaller LR than the randomly-init head
LR_BACKBONE  = 2e-5
LR_HEAD      = 2e-4
WEIGHT_DECAY = 1e-4

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training on:', DEVICE)

#Class imbalance weights (capped at 50)
POS_WEIGHT = torch.tensor([
     9.4496, 49.6807, 29.3338, 50.0000,  8.9910,
    50.0000, 50.0000, 50.0000,  5.2785, 20.4563,
     0.7134, 17.3755, 37.5870, 50.0000, 31.8070
], dtype=torch.float32).to(DEVICE)

## Model — DenseNet-121 + Clinical Branch
Fuses image features (1024-dim) with age+gender (16-dim) before the final classifier.

In [ ]:
import torch.nn as nn
from torchvision import models

class ChestXrayModel(nn.Module):
    def __init__(self, num_classes=15, pretrained=True):
        super().__init__()
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        densenet = models.densenet121(weights=weights)

        self.features = densenet.features         
        self.gap      = nn.AdaptiveAvgPool2d((1, 1))

        # small branch for age + gender (2 inputs -> 16 features)
        self.clinical_branch = nn.Sequential(
            nn.Linear(2, 16), nn.ReLU(), nn.Dropout(0.3)
        )

        # final classifier: 1024 image features + 16 clinical features
        self.classifier = nn.Sequential(
            nn.Linear(1024 + 16, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, image, age, gender):
        x=torch.flatten(self.gap(self.features(image)), 1)  # (B, 1024)
        clinical=self.clinical_branch(torch.cat([age, gender], dim=1))  # (B, 16)
        return self.classifier(torch.cat([x, clinical], dim=1))           # (B, 15)


# quick shape check
with torch.no_grad():
    _m = ChestXrayModel(pretrained=False)
    _o = _m(torch.randn(2,3,224,224), torch.randn(2,1), torch.randn(2,1))
    assert _o.shape == (2, 15), f'Bad output shape: {_o.shape}'
    del _m
print('Model definition OK — output shape (2, 15) confirmed')

## Dataset & Transforms
Improved augmentation vs the original: `RandomAffine` replaces bare rotation (adds translation + scale), and `RandomErasing` randomly hides a small patch so the model learns to use the full lung field.

In [ ]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration',
    'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening',
    'Pneumonia', 'Pneumothorax'
]

# NIH dataset stores images across 12 sub-folders
IMAGE_DIRS = [f'images_{str(i).zfill(3)}/images' for i in range(1, 13)]


class ChestXrayDataset(Dataset):
    def __init__(self, csv_path, image_root, transform=None):
        self.df         = pd.read_csv(csv_path)
        self.image_root = image_root
        self.transform  = transform

    def __len__(self):
        return len(self.df)

    def _find_image(self, filename):
        for folder in IMAGE_DIRS:
            path = os.path.join(self.image_root, folder, filename)
            if os.path.exists(path):
                return path
        raise FileNotFoundError(f'Image not found: {filename}')

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(self._find_image(row['Image Index'])).convert('RGB')
        if self.transform:
            image = self.transform(image)

        target = torch.zeros(len(LABELS), dtype=torch.float32)
        for i, label in enumerate(LABELS):
            if label in row['Finding Labels']:
                target[i] = 1.0

        # age and gender are already preprocessed in the CSV:
        # age / 100.0,  gender: M=0 F=1
        age    = torch.tensor([row['Patient Age']],    dtype=torch.float32)
        gender = torch.tensor([row['Patient Gender']], dtype=torch.float32)
        return image, age, gender, target


def get_transforms(train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            # RandomAffine combines rotation + small translation + scale in one pass
            transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                  std=[0.229, 0.224, 0.225]),
            # randomly erase a small patch — forces model to use full lung field
            transforms.RandomErasing(p=0.3, scale=(0.02, 0.08)),
        ])
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                              std=[0.229, 0.224, 0.225]),
    ])


print('Dataset and transforms defined')

## Verify Paths

In [ ]:
print('CSV exists   :', os.path.exists(TRAIN_CSV))
print('Images exist :', os.path.exists(IMAGE_ROOT))
print()
print('Image folders found:')
for d in os.listdir(IMAGE_ROOT):
    if d.startswith('images_'):
        print(' ', d)

## DataLoader — Train / Val Split
Two dataset instances from the same CSV — one with augmentation (train), one without (val) — then split by the same fixed indices so there's no data leakage.

In [ ]:
from torch.utils.data import DataLoader, Subset

def get_loaders():
    train_full = ChestXrayDataset(TRAIN_CSV, IMAGE_ROOT, transform=get_transforms(train=True))
    val_full   = ChestXrayDataset(TRAIN_CSV, IMAGE_ROOT, transform=get_transforms(train=False))

    n     = len(train_full)
    val_n = int(n * VAL_SPLIT)
    rng   = np.random.default_rng(42)
    idx   = rng.permutation(n).tolist()
    train_idx, val_idx = idx[val_n:], idx[:val_n]

    # val batch can be 2x larger because there's no gradient computation
    train_loader = DataLoader(Subset(train_full, train_idx),
                              batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              persistent_workers=True, prefetch_factor=2)
    val_loader   = DataLoader(Subset(val_full, val_idx),
                              batch_size=BATCH_SIZE * 2, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              persistent_workers=True, prefetch_factor=2)
    return train_loader, val_loader

train_loader, val_loader = get_loaders()
print(f'Train samples : {len(train_loader.dataset):,}')
print(f'Val samples   : {len(val_loader.dataset):,}')
print(f'Train batches : {len(train_loader):,}')

## Model, Optimizer & Scheduler
- **AdamW** with weight decay for regularization (Adam had none)
- **Differential LRs**: backbone gets 10× smaller LR than the head — preserves pretrained features while letting the new classifier layers learn fast
- **OneCycleLR**: 20% warmup then cosine annealing — consistently reaches better minima than ReduceLROnPlateau in a fixed number of epochs

In [ ]:
model     = ChestXrayModel(num_classes=15, pretrained=True).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)

optimizer = torch.optim.AdamW([
    {'params': model.features.parameters(),                         'lr': LR_BACKBONE},
    {'params': list(model.clinical_branch.parameters()) +
               list(model.classifier.parameters()),                 'lr': LR_HEAD},
], weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[LR_BACKBONE, LR_HEAD],
    epochs=NUM_EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.2,   # 20% warmup, 80% cosine cooldown
)

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total:,}')

## Resume from Checkpoint *(optional)*
Skip this cell if training from scratch. Set `RESUME_CHECKPOINT` to `'model_best.pth'` or `'model_epoch20.pth'` etc.

In [ ]:
RESUME_CHECKPOINT = None   
START_EPOCH = 0
best_auc    = 0.0

if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
    ckpt = torch.load(RESUME_CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if 'scheduler_state_dict' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    START_EPOCH = ckpt['epoch']
    best_auc    = ckpt.get('val_auc', 0.0)
    print(f'Resumed from epoch {START_EPOCH}, val_AUC {best_auc:.4f}')
else:
    print('Starting from scratch')

## Training Loop
Each epoch: train pass → val pass → print metrics → save best checkpoint.

Key additions vs the original:
- **Validation loop** with AUC-ROC 
- **Gradient clipping** (`max_norm=1.0`) — high pos_weights can spike the loss and blow up gradients
- **`scheduler.step()` every batch** — required by OneCycleLR
- **Best checkpoint** saved by val AUC, not just every 5 epochs
- Fixed deprecated `torch.cuda.amp` → `torch.amp`

In [ ]:
from torch.amp import GradScaler, autocast
from sklearn.metrics import roc_auc_score
from tqdm.notebook import tqdm

scaler   = GradScaler('cuda')
history  = []
best_auc = best_auc if 'best_auc' in dir() else 0.0


def mean_auc(targets, probs):
    """Mean AUC-ROC across all classes that have at least one positive sample."""
    aucs = []
    for i in range(targets.shape[1]):
        if targets[:, i].sum() > 0:
            aucs.append(roc_auc_score(targets[:, i], probs[:, i]))
    return float(np.mean(aucs)) if aucs else 0.0


def save_checkpoint(epoch, val_loss, val_auc, tag):
    path = os.path.join(CHECKPOINT_DIR, f'model_{tag}.pth')
    torch.save({
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict':    scaler.state_dict(),
        'val_loss':             val_loss,
        'val_auc':              val_auc,
        'history':              history,
    }, path)
    return path


try:
    for epoch in range(START_EPOCH, NUM_EPOCHS):

        # Train 
        model.train()
        train_loss = 0.0

        for images, ages, genders, targets in tqdm(train_loader,
                                                    desc=f'Epoch {epoch+1:02d}/{NUM_EPOCHS} train'):
            images, ages, genders, targets = (
                images.to(DEVICE, non_blocking=True),
                ages.to(DEVICE, non_blocking=True),
                genders.to(DEVICE, non_blocking=True),
                targets.to(DEVICE, non_blocking=True),
            )
            optimizer.zero_grad()

            with autocast('cuda'):
                loss = criterion(model(images, ages, genders), targets)

            scaler.scale(loss).backward()
            # clip gradients — high pos_weights can spike the loss -> exploding gradients
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()   # OneCycleLR must step every batch
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validate
        model.eval()
        val_loss, all_probs, all_targets = 0.0, [], []

        with torch.no_grad():
            for images, ages, genders, targets in tqdm(val_loader,
                                                        desc=f'Epoch {epoch+1:02d}/{NUM_EPOCHS} val  '):
                images, ages, genders, targets = (
                    images.to(DEVICE, non_blocking=True),
                    ages.to(DEVICE, non_blocking=True),
                    genders.to(DEVICE, non_blocking=True),
                    targets.to(DEVICE, non_blocking=True),
                )
                with autocast('cuda'):
                    logits = model(images, ages, genders)
                    val_loss += criterion(logits, targets).item()
                all_probs.append(torch.sigmoid(logits).cpu().numpy())
                all_targets.append(targets.cpu().numpy())

        val_loss /= len(val_loader)
        val_auc   = mean_auc(np.concatenate(all_targets), np.concatenate(all_probs))

        history.append({'epoch': epoch+1, 'train_loss': train_loss,
                        'val_loss': val_loss, 'val_auc': val_auc})
        print(f'Epoch {epoch+1:02d} | '
              f'train_loss {train_loss:.4f} | '
              f'val_loss {val_loss:.4f} | '
              f'val_AUC {val_auc:.4f}')

        # save best checkpoint by val AUC
        if val_auc > best_auc:
            best_auc = val_auc
            save_checkpoint(epoch+1, val_loss, val_auc, 'best')
            print(f'  best model saved (AUC {best_auc:.4f})')

        # periodic checkpoint every 5 epochs
        if (epoch + 1) % 5 == 0:
            save_checkpoint(epoch+1, val_loss, val_auc, f'epoch{epoch+1}')

except KeyboardInterrupt:
    print('\nInterrupted — saving checkpoint...')
    save_checkpoint(epoch+1,
                    val_loss if 'val_loss' in dir() else -1,
                    val_auc  if 'val_auc'  in dir() else 0.0,
                    'interrupt')
    print('Set RESUME_CHECKPOINT above and re-run from the DataLoader cell to continue.')

except Exception as e:
    print(f'Crashed: {e}')
    save_checkpoint(epoch+1, -1, 0.0, 'crash')
    raise

else:
    print(f'\nTraining complete. Best val AUC: {best_auc:.4f}')

## Download Checkpoints

In [ ]:
import os
print('Files in checkpoint dir:')
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    size_mb = os.path.getsize(os.path.join(CHECKPOINT_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/checkpoints', 'zip', CHECKPOINT_DIR)
print('Zipped to /kaggle/working/checkpoints.zip')

In [ ]:
from IPython.display import FileLink
FileLink('checkpoints.zip')

In [ ]:
final_path = os.path.join(CHECKPOINT_DIR, 'model_final.pth')
torch.save(model.state_dict(), final_path)
print(f'Final model saved -> {final_path}')

## Metrics Plot
Run after training completes (or at any point to see progress so far).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame(history)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(df['epoch'], df['train_loss'], marker='o', label='train', linewidth=2, color='steelblue')
ax1.plot(df['epoch'], df['val_loss'],   marker='o', label='val',   linewidth=2, color='orange')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCE Loss'); ax1.set_title('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(df['epoch'], df['val_auc'], marker='o', linewidth=2, color='green')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('AUC-ROC'); ax2.set_title('Validation AUC')
ax2.set_ylim(0, 1); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'metrics.png'), dpi=150)
plt.show()
print(f'Best val AUC so far: {max(df["val_auc"]):.4f}')